In [1]:
import torch
import torch.nn as nn
from torch import Tensor
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader,SubsetRandomSampler,ConcatDataset
import torch.nn.functional as F

In [2]:
import sys
import os
sys.path.append(os.path.abspath("/home1/smaruj/pytorch_akita/"))

# from model import SeqNN
from model_v2_compatible import SeqNN

In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(device)

cuda:0


In [4]:
model = SeqNN()
model.load_state_dict(torch.load("/home1/smaruj/pytorch_akita/model_0_v2_finetuned_correctly.pt", map_location=device))
model.eval()

/tmp/SLURM_1182541/ipykernel_2145444/2744654124.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("/home1/smaruj/pytorch_akita/model_0_v2_

SeqNN(
  (stochastic_reverse_complement): StochasticReverseComplement()
  (stochastic_shift): StochasticShift()
  (conv_block_1): ConvBlock(
    (conv): Conv1d(4, 128, kernel_size=(15,), stride=(1,), padding=(7,), bias=False)
    (batch_norm): BatchNorm1d(128, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_tower): ConvTower(
    (conv_tower): Sequential(
      (0): ReLU()
      (1): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): ReLU()
      (5): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (6): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (7): MaxPool1d(kernel_size=2, stride=2, paddi

In [ ]:
# Load the entire model (architecture + weights)
# model = torch.load("/home1/smaruj/pytorch_akita/model.pth")

In [ ]:
# model = model.to(device)

In [ ]:
# Set the model to evaluation mode (important for inference)
# model.eval()

In [5]:
from torchinfo import summary

summary(model, input_size=(2, 4, 1310720), col_names=["output_size", "num_params"])

Layer (type:depth-idx)                   Output Shape              Param #
SeqNN                                    [2, 1, 130305]            --
├─StochasticReverseComplement: 1-1       [2, 4, 1310720]           --
├─StochasticShift: 1-2                   [2, 4, 1310720]           --
├─ConvBlock: 1-3                         [2, 128, 655360]          --
│    └─Conv1d: 2-1                       [2, 128, 1310720]         7,680
│    └─BatchNorm1d: 2-2                  [2, 128, 1310720]         256
│    └─MaxPool1d: 2-3                    [2, 128, 655360]          --
├─ConvTower: 1-4                         [2, 128, 640]             --
│    └─Sequential: 2-4                   [2, 128, 640]             --
│    │    └─ReLU: 3-1                    [2, 128, 655360]          --
│    │    └─Conv1d: 3-2                  [2, 128, 655360]          81,920
│    │    └─BatchNorm1d: 3-3             [2, 128, 655360]          256
│    │    └─MaxPool1d: 3-4               [2, 128, 327680]          --
│    │

In [6]:
import pandas as pd

In [7]:
FOLD = 0
input_dir = "/scratch1/smaruj/genomic_map_transformation"

In [8]:
df = pd.read_csv(f"{input_dir}/df_select_fold{FOLD}.tsv", sep="\t")

In [9]:
df = df[:10]

In [10]:
df['target_chrom'] = df['chrom'].shift(-1)
df['target_start'] = df['start'].shift(-1)
df['target_end'] = df['end'].shift(-1)

In [11]:
# Fill last row with values from the first row
df.loc[df.index[-1], 'target_chrom'] = df.loc[df.index[0], 'chrom']
df.loc[df.index[-1], 'target_start'] = df.loc[df.index[0], 'start']
df.loc[df.index[-1], 'target_end'] = df.loc[df.index[0], 'end']

In [12]:
# Convert to int
df['target_start'] = df['target_start'].astype(int)
df['target_end'] = df['target_end'].astype(int)

In [ ]:
df

In [13]:
# to ensure the local, forked ledidi is used
# not the one installed using pip

import sys
sys.path.insert(0, "/home1/smaruj/ledidi/ledidi/")  # Add the directory where "ledidi" is located
from ledidi_whole_seq import Ledidi

In [14]:
df["last_accepted_step"] = -1  # initialize with placeholder

In [ ]:
df

In [15]:
for i, row in enumerate(df[:1].itertuples(index=False)):
    chrom, pred_start, pred_end = row.chrom, row.start, row.end
    target_chrom, target_start, target_end = row.target_chrom, row.target_start, row.target_end
    
    print(f"Map transformation starting from genome location: {chrom}:{pred_start}-{pred_end}")
    print(f"To a genome location: {target_chrom}:{target_start}-{target_end}")
    
    X = torch.load(f"/scratch1/smaruj/genomic_map_transformation/ohe_X_fold0/{chrom}_{pred_start}_{pred_end}_X.pt", weights_only=True, map_location=device)
    target = torch.load(f"/scratch1/smaruj/genomic_map_transformation/genomic_targets_fold0/{target_chrom}_{target_start}_{target_end}_target.pt", weights_only=True, map_location=device)

    wrapper = Ledidi(model, 
                    input_loss=torch.nn.L1Loss(reduction='sum'), 
                    output_loss=torch.nn.L1Loss(reduction='sum'),   # output_loss=torch.nn.MSELoss(),
                    batch_size=1,
                    l=0.05,
                    max_iter=300,
                    early_stopping_iter=300,
                    return_history=False,
                    verbose=True
                    ).cuda()
    
    generated_seq, last_update = wrapper.fit_transform(X, target)
    
    # Update df with last_accepted_step
    df.at[i, "last_accepted_step"] = last_update
    
    # torch.save(generated_seq, f"/scratch1/smaruj/genomic_insertion_loci/genomic_optimization_results/fold{FOLD}/{chrom}_{pred_start}_{pred_end}_seq.pt")

# df.to_csv(f"/scratch1/smaruj/genomic_insertion_loci/genomic_optimization_results/fold{FOLD}_test.tsv", sep="\t", index=False)

Map transformation starting from genome location: chr5:63203328-64514048
To a genome location: chr3:138672128-139982848
Model in train mode: False
Gradients enabled for weights: True
Weights shape: torch.Size([1, 4, 1310720])
y_hat shape torch.Size([1, 1, 130305])
y_bar shape torch.Size([1, 1, 130305])
Global loss applied.
X_ shape torch.Size([1, 4, 1310720])
y_bar shape torch.Size([1, 1, 130305])
iter=I	input_loss=0.0	output_loss=2.426e+04	total_loss=2.426e+04	time=0.0
iter=100	input_loss=1.37e+05	output_loss=6.15e+03	total_loss=1.3e+04	time=9.915
iter=200	input_loss=1.319e+05	output_loss=4.653e+03	total_loss=1.125e+04	time=9.536
iter=300	input_loss=1.503e+05	output_loss=5.372e+03	total_loss=1.289e+04	time=9.635
iter=F	input_loss=1.329e+05	output_loss=3.48e+03	total_loss=1.013e+04	time=29.09
Last iteration with update:  206


In [ ]:
# slice_torch = X[:,:,start_index : end_index]

In [ ]:
# X_l_flank = X[:,:,start_index - 2048*2 : start_index]
# X_r_flank = X[:,:,end_index : end_index + 2048*2]

In [ ]:
# x_bar, history = wrapper.fit_transform(X, modified_vector_tensor)
# if use_semifreddo=True
# x_bar = wrapper.fit_transform(slice_torch, y_bar, X_l_flank=X_l_flank, X_r_flank=X_r_flank)
# x_bar, history = wrapper.fit_transform(slice_torch, y_bar, X_l_flank=X_l_flank, X_r_flank=X_r_flank)

In [ ]:
# if use_semifreddo=False
# x_bar = wrapper.fit_transform(X, y_bar)
x_bar, history = wrapper.fit_transform(X, y_bar)

In [ ]:
x_bar

## Input and Output Loss Plots

In [ ]:
%matplotlib inline
import numpy
import matplotlib.pyplot as plt
import seaborn; seaborn.set_style('whitegrid')

plt.figure(figsize=(5, 3))
plt.plot(history['input_loss'], c='0.7', label="Input Loss")
plt.legend(fontsize=10)
plt.xlabel("Iterations")
plt.ylabel("Loss")

seaborn.despine(left=True, bottom=True)
plt.show()

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(history['output_loss'], c='0.3', label="Output Loss")
plt.legend(fontsize=10)
plt.xlabel("Iterations")
plt.ylabel("Loss")

seaborn.despine(left=True, bottom=True)
plt.show()

In [ ]:
seq1_indices = torch.argmax(x_bar, dim=1)
seq2_indices = torch.argmax(X, dim=1)

In [ ]:
num_differences = (seq1_indices != seq2_indices).sum().item()

print(f"Number of differing nucleotides: {num_differences}")

## Predicted Maps

In [ ]:
model.eval()
with torch.no_grad():
    pred = model(generated_seq)
    pred_back = model(X)

In [ ]:
pred.shape, pred_back.shape

In [ ]:
from scipy.stats import pearsonr

# Flatten tensors
x_flat = pred.view(-1).cpu().numpy()
y_flat = target.view(-1).cpu().numpy()

# Calculate Pearson R
r_value, _ = pearsonr(x_flat, y_flat)
print(f"Pearson R: {r_value}")

In [ ]:
import numpy as np

In [ ]:
# Helper function to set diagonal elements to a specific value
def set_diag(matrix, value, k):
    # Explicitly set the diagonal to 'value' (in this case, np.nan) for each k
    rows, cols = matrix.shape
    for i in range(rows):
        if 0 <= i + k < cols:
            matrix[i, i + k] = value


def from_upper_triu(vector_repr, matrix_len, num_diags):
    # Ensure vector_repr is a NumPy array (if it's a PyTorch tensor, convert it)
    if isinstance(vector_repr, torch.Tensor):
        vector_repr = vector_repr.detach().flatten().cpu().numpy()  # Flatten and convert to NumPy array

    # Initialize a zero matrix of shape (matrix_len, matrix_len)
    z = np.zeros((matrix_len, matrix_len))

    # Get the indices for the upper triangular matrix
    triu_tup = np.triu_indices(matrix_len, num_diags)

    # Assign the values from the vector_repr to the upper triangular part of the matrix
    z[triu_tup] = vector_repr

    # Set the diagonals specified by num_diags to np.nan
    for i in range(-num_diags + 1, num_diags):
        set_diag(z, np.nan, i)

    # Ensure the matrix is symmetric
    return z + z.T

In [ ]:
matrix_to_plot = from_upper_triu(pred[0, 0, :], matrix_len=512, num_diags=2)

In [ ]:
matrix_og_to_plot = from_upper_triu(pred_back[0, 0, :], matrix_len=512, num_diags=2)

In [ ]:
matrix_to_plot_genomic = from_upper_triu(target[0, 0, :], matrix_len=512, num_diags=2)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
width=5
height=5
vmin = -0.6
vmax = 0.6
palette="RdBu_r"

fig, axes = plt.subplots(1, 1, figsize=(width, height))

sns.heatmap(
    matrix_og_to_plot,
    vmin=vmin,
    vmax=vmax,
    cbar=True,
    cmap=palette,
    square=True,
    xticklabels=False,
    yticklabels=False,
    ax=axes
)

# # Highlight the 194th row
# axes.hlines(y=256, xmin=0, xmax=512, colors='black', linewidth=1.5)

# # Highlight the 254th column
# axes.vlines(x=256, ymin=0, ymax=512, colors='black', linewidth=1.5)

axes.set_title("Original")

plt.tight_layout()
plt.show()

In [ ]:
width=5
height=5
vmin = -0.6
vmax = 0.6
palette="RdBu_r"

fig, axes = plt.subplots(1, 1, figsize=(width, height))

sns.heatmap(
    matrix_to_plot_genomic,
    vmin=vmin,
    vmax=vmax,
    cbar=True,
    cmap=palette,
    square=True,
    xticklabels=False,
    yticklabels=False,
    ax=axes
)

# # Highlight the 194th row
# axes.hlines(y=256, xmin=0, xmax=512, colors='black', linewidth=1.5)

# # Highlight the 254th column
# axes.vlines(x=256, ymin=0, ymax=512, colors='black', linewidth=1.5)

axes.set_title("Target")

plt.tight_layout()
plt.show()

In [ ]:
width=5
height=5
vmin = -0.6
vmax = 0.6
palette="RdBu_r"

fig, axes = plt.subplots(1, 1, figsize=(width, height))

sns.heatmap(
    matrix_to_plot,
    vmin=vmin,
    vmax=vmax,
    cbar=True,
    cmap=palette,
    square=True,
    xticklabels=False,
    yticklabels=False,
    ax=axes
)

# # Highlight the 194th row
# axes.hlines(y=256, xmin=0, xmax=512, colors='black', linewidth=1.5)

# # Highlight the 254th column
# axes.vlines(x=256, ymin=0, ymax=512, colors='black', linewidth=1.5)

axes.set_title("Result")

plt.tight_layout()
plt.show()